# Colab Macro-Policy BC Training

This notebook is the only supported training path for free Google Colab in this repo.

It assumes you already produced a compact macro dataset locally and uploaded that single file to Google Drive:

- `macro_dataset.jsonl.gz`

Do not upload raw per-step trajectory JSON to Colab.

Recommended local workflow:

1. Collect episodes locally into `macro_dataset.jsonl.gz`.
2. Upload only that compact dataset to Google Drive.
3. Delete local temporary artifacts if you no longer need them for debugging.

For free Colab, use a **small Qwen-family model**. The default below is `Qwen/Qwen2.5-0.5B-Instruct`, because that is realistic on a free T4. If you have a small enough Qwen 3.5 checkpoint, replace `MODEL_NAME` in the config cell.

## Local dataset build step

Collect the compact dataset locally before using Colab:

```bash
python scripts/collect_bot_data.py \
    --episodes 50 \
    --bot normal \
    --dataset-path data/macro/macro_dataset.jsonl.gz
```

Then upload `macro_dataset.jsonl.gz` to Google Drive, for example:

```text
MyDrive/openra/macro_dataset.jsonl.gz
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

REPO_URL = 'https://github.com/huixu11/openra-rl-challenge.git'
REPO_DIR = '/content/openra-rl-challenge'

if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR

In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q transformers datasets trl peft accelerate bitsandbytes pandas sentencepiece

In [ ]:
import os
import torch

DATA_PATH = '/content/drive/MyDrive/openra/macro_dataset.jsonl.gz'
OUTPUT_DIR = '/content/drive/MyDrive/openra/checkpoints/openra-bc-qwen-colab'

# Free Colab default: keep this small.
MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

# Conservative defaults for free Colab.
MAX_ROWS = 50000
EPOCHS = 1
MAX_SEQ_LENGTH = 512
BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRAD_ACCUM = 16
VAL_RATIO = 0.05

assert os.path.exists(DATA_PATH), f'Missing dataset: {DATA_PATH}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU detected. In Colab, switch Runtime -> Change runtime type -> T4 GPU.')

print('Dataset:', DATA_PATH)
print('Output:', OUTPUT_DIR)
print('Model:', MODEL_NAME)

## Dataset sufficiency check

The goal here is not just file size. What matters is whether the compact dataset contains enough rows and enough unique episodes.

Practical thresholds:

- fewer than `20` episodes: too small, mostly a smoke test
- `20-49` episodes: usable for a first experiment
- `50+` episodes: good starting point
- fewer than `20k` rows: too small for a serious run
- `50k+` rows: good Colab starting point
- `100k+` rows: much stronger if the model still fits

In [ ]:
import gzip
import json
from collections import Counter

row_count = 0
episodes = set()
intent_counts = Counter()
phase_counts = Counter()

with gzip.open(DATA_PATH, 'rt', encoding='utf-8-sig') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        row_count += 1
        episodes.add(row.get('episode', ''))
        intent_counts[row.get('primary_intent', 'unknown')] += 1
        phase_counts[row.get('phase', 'unknown')] += 1

episode_count = len(episodes)
print('Rows:', row_count)
print('Episodes:', episode_count)
print('Top intents:', intent_counts.most_common(10))
print('Top phases:', phase_counts.most_common(10))

if episode_count < 20 or row_count < 20000:
    print('\nStatus: TOO SMALL for a serious training run.')
elif episode_count < 50 or row_count < 50000:
    print('\nStatus: OK for a first experiment, but collect more if results are weak.')
else:
    print('\nStatus: ENOUGH to start macro-policy BC training on Colab.')

In [ ]:
import subprocess

prepare_cmd = [
    'python', 'scripts/train_bc_qwen.py',
    '--data-path', DATA_PATH,
    '--prepare-only',
    '--max-rows', str(MAX_ROWS),
    '--val-ratio', str(VAL_RATIO),
]

print(' '.join(prepare_cmd))
subprocess.run(prepare_cmd, check=True)

## Start training

This uses:

- 4-bit loading
- LoRA adapters
- gradient checkpointing
- small batch size with accumulation

Those settings are chosen specifically for the free Colab constraint.

In [ ]:
import subprocess

train_cmd = [
    'python', 'scripts/train_bc_qwen.py',
    '--data-path', DATA_PATH,
    '--model', MODEL_NAME,
    '--output-dir', OUTPUT_DIR,
    '--load-in-4bit',
    '--gradient-checkpointing',
    '--batch-size', str(BATCH_SIZE),
    '--eval-batch-size', str(EVAL_BATCH_SIZE),
    '--grad-accum', str(GRAD_ACCUM),
    '--max-seq-length', str(MAX_SEQ_LENGTH),
    '--epochs', str(EPOCHS),
    '--max-rows', str(MAX_ROWS),
    '--val-ratio', str(VAL_RATIO),
    '--save-steps', '200',
    '--eval-steps', '100',
]

print(' '.join(train_cmd))
subprocess.run(train_cmd, check=True)

In [ ]:
import os

for root, dirs, files in os.walk(OUTPUT_DIR):
    depth = root.replace(OUTPUT_DIR, '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root) or root}')
    for file_name in sorted(files):
        print(f'{indent}  {file_name}')

## Notes

- If you hit out-of-memory errors, reduce `MAX_ROWS`, `MAX_SEQ_LENGTH`, or use an even smaller model.
- If training is stable, scale up data gradually instead of starting with the full dataset.
- This notebook is intentionally macro-policy only.